In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

In [2]:
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
#from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

import sys
sys.path.append('../../ss_lal_military/src')
sys.path.append('../src/')
sys.path.append('../migrant/notebooks/utilities/')

In [3]:
import os
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_agent_full_script_new_node', 3) # For example, MyPySpark3
  
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v24, Python_path: /data/sdp/mlpy3811v24/bin/python, Resource_level: 3.middle(81,588)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 12:32:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/05 12:32:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/05 12:32:39 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/05 12:32:42 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
26/03/05 12:32:43 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Client$Connection.setupSaslConnection(Client.java:623)
	at org.apache.hadoop.ipc.Client$Connection.acces

In [4]:
company_id = [
        '114177','114392','114391','114381','114389','114383','114388','114386',
    '114376','114369','114364','114372','114356','114357','114375','114368','114367',
    '114102','114126','114047','114177','113704','114102','114126','114047','114177',
    '113704','114102','112711','112713','113245','112698','112714','112715','113386',
    '112457','113528','112448','112455','112716','112708','112710','112454','112696',
    '113704','112695','112691','113585','112678','112697','112690','113519','113491',
    '112694','112693','113242','112014','112073','112075','112095','112100','112457',
    '112465','112455','112454','112071','112220','112176','112085','112617','112089',
    '112218','112012','112072','112079','112078','112088','112205','111007','111015',
    '111010','111025','111019','111041','111021','111298','110774','111164','111014',
    '111046','111042','111009','111004','111510','111699','111011','111007','110418',
    '111015','111010','110417','110422','110383','111019','110421','111041','110518',
    '110774','111014','110500','111046','110519','111042','110403','111004','110503',
    '110947','110410','110502','110347','110382','110412','110406','111011','110405',
    '109555','109522','109754','109571','109502','109541','109597','109503','109755',
    '109528','109524','109501','108895','108667','107947','108326','108138','107774',
    '107055','106949','107024'
]

In [5]:
company_name = [
# I can not show name of companies names, because i am under NDA
]

In [6]:
"""Функции для взаимодействия с Greenplum."""

import getpass
import pandas as pd
import sqlalchemy
from sqlalchemy import text


def get_sqlalchemy_engine():
    parameters = {
        "user": getpass.getuser().split("_")[0],
        "host": "gp_dns_pkap1150.gp.df.sbrf.ru",
        "port": 5432,
        "database": "gp_rozn2",
    }

    url = "postgresql+psycopg2://{user}@{host}:{port}/{database}".format(**parameters)
    engine = sqlalchemy.create_engine(url=url)

    return engine

class greenplum_con():
    """GP подключение"""
    def __init__(self):
        self.connection = get_sqlalchemy_engine().connect()
        
    def take(self, query):
        try:
            data = pd.read_sql_query(text(query), self.connection)
            return data
        except Exception as e:
            self.close()
            self.connect()
            print(str(e))
    
    def connect(self):
        self.connection = get_sqlalchemy_engine().connect()
        
    def close(self):
        self.connection.close()

def print_execution_plan(query: str, engine) -> None:
    """Напечатать план выполнения запроса.

    Перед запуском потенциально тяжелых запросов рекомендуется проверять план
    выполнения. Стоимость не должна превышать "сотни тысяч - миллионы".

    """

    plan = engine.execute(f"EXPLAIN {query}").fetchall()
    plan = "\n".join(map(lambda x: x[0], plan))
    print(plan)

In [7]:
gp = greenplum_con()

In [8]:
engine = get_sqlalchemy_engine()
con = engine.connect()

In [11]:
df_target_0_first_half = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_zero_new_iter_first_half_new
'''), con )

In [16]:
df_target_0_first_half_old = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_zero_new_iter_first_half
'''), con )

In [17]:
df_target_0_second_half_old = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_zero_new_iter_second_half
'''), con )

In [18]:
df_target_0_old = pd.concat([df_target_0_first_half_old, df_target_0_second_half_old], axis=0)

In [9]:
df_target_0_second_half = pd.read_sql(text(f'''
select a.epk_id, a.start_dt, a.end_dt, unique_only_nflag, campaign_name
    from s_grnplm_ld_rozn_electron_aaas_dm.evk_hist a 
    join s_grnplm_ld_rozn_electron_aaas_dm.evk_dic b on a.sk_id = b.sk_id
    left join s_grnplm_ld_rozn_electron_aaas_dm.gl_evo_resp_cln_month c on a.epk_id = c.epk_id and b.ab_test_id = c.ab_test_id and resp_month >= '2025-08-01' and resp_month <= '2025-10-31'
    where a.start_dt >= '2025-08-01' and a.end_dt <= '2025-10-31' 
    and b.start_dt >= '2025-08-01' and b.end_dt <= '2025-10-31' and (unique_only_nflag = 0 or unique_only_nflag is null) 
    and campaign_name in (# I can not show name of companies names, because i am under NDA
)
order by random()
limit 30618
'''), con )

In [12]:
df_target_0 = pd.concat([df_target_0_first_half, df_target_0_second_half], axis=0)

In [20]:
df_target_0_sample = df_target_0.sample(n=23489, random_state=0)
df_target_0_new = pd.concat([df_target_0_old, df_target_0_sample], ignore_index=True)

In [22]:
df_target_0_new.to_parquet('df_target_0_new.parquet')

In [23]:
df_target_1_first_half = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_one_new_iter_first_half
'''), con )

In [24]:
df_target_1_second_half = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_one_new_iter_second_half
'''), con )

In [25]:
df_target_1 = pd.concat([df_target_1_first_half, df_target_1_second_half], axis=0)

In [ ]:
df_target_1.to_parquet('df_target_1_new.parquet')

In [27]:
df_target = pd.concat([df_target_1, df_target_0_new], axis=0)

In [29]:
df_target.to_parquet('target_new.parquet')